# LionAG2: Recursive Exploratory Research with AG2 beta — wiring agents together (3/10)

In the [last tutorial](02_typed_findings.ipynb), we got our agent to produce a typed `Finding` object — a Pydantic model with topic, summary, citations, and a novelty score. That's useful on its own, but the real power shows up when one agent's structured output becomes another agent's input.

Today we wire two agents into a pipeline: a **Surveyor** maps the territory and a **Theorist** picks the most interesting open question and turns it into a falsifiable hypothesis. The handoff is just Python — serialize the typed object into the next agent's prompt.

## Setup

In [1]:
import os

from dotenv import load_dotenv
from IPython.display import Markdown, display
from pydantic import BaseModel, Field

from autogen.beta import Agent
from autogen.beta.config import OpenAIConfig
from autogen.beta.tools import ExaToolkit

load_dotenv()

config = OpenAIConfig(
    model="gpt-5.4-mini",
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url="https://api.openai.com/v1",
)
exa_tool = ExaToolkit(api_key=os.getenv("EXA_API_KEY"))

## Richer schemas for a research pipeline

Last time we had a single `Finding`. A research pipeline needs more structure — the surveyor should tell us *what's known* and *what's open*, and the theorist should produce something *testable*.

In [2]:
class OpenQuestion(BaseModel):
    question: str = Field(description="A specific unresolved question surfaced by the survey.")
    novelty: float = Field(
        ge=0.0, le=1.0,
        description="0 = well-studied, 1 = barely explored.",
    )


class Survey(BaseModel):
    """Surveyor output: landscape of a topic plus open frontiers."""
    topic: str
    overview: str = Field(description="2-3 sentence summary of the current state of knowledge.")
    key_findings: list[str] = Field(description="Major established results (3-5 bullets).")
    open_questions: list[OpenQuestion] = Field(
        description="Unresolved questions ranked by novelty. At least 2.",
    )

The `OpenQuestion` carries a `novelty` score — this is what will drive recursive depth in later tutorials. For now, the theorist just picks the most novel one.

In [3]:
class Hypothesis(BaseModel):
    """Theorist output: one falsifiable claim with a test."""
    claim: str = Field(description="One-sentence falsifiable claim.")
    mechanism: str = Field(description="Why this claim might hold — the causal story.")
    testable_prediction: str = Field(
        description="An observable that would confirm or refute the claim."
    )
    confidence: float = Field(ge=0.0, le=1.0, description="Subjective confidence in the claim.")
    source_question: str = Field(description="The open question this hypothesis addresses.")

## Agent 1: the Surveyor

The surveyor's job is to map the territory. We give it Exa tools so it can actually search, and `response_schema=Survey` so its output is typed.

In [4]:
surveyor = Agent(
    name="surveyor",
    prompt=(
        "You are a meticulous research surveyor. Search broadly, then "
        "summarize what is known and what remains open. Be specific about "
        "novelty — textbook material is 0.0, active frontiers are 0.7+."
    ),
    config=config,
    response_schema=Survey,
    tools=[exa_tool],
)

survey_reply = await surveyor.ask(
    "Survey the current state of high-Tc superconductivity research."
)
survey: Survey = await survey_reply.content()

In [5]:
lines = [f"## Survey: {survey.topic}", "", survey.overview, "", "**Key findings:**"]
for kf in survey.key_findings:
    lines.append(f"- {kf}")
lines.append("")
lines.append("**Open questions:**")
for q in survey.open_questions:
    lines.append(f"- [{q.novelty:.2f}] {q.question}")
display(Markdown("\n".join(lines)))

## Survey: High-Tc superconductivity research

High-Tc research in 2024–2025 is no longer just about cuprates: the field now spans cuprates, iron-based superconductors, and a newly active nickelate family, with hydrides as a separate high-pressure route to very high Tc. For cuprates, the main advances are not in finding a mechanism but in sharpening the phenomenology of the pseudogap, charge order/stripes, pair-density-wave (PDW) physics, and strange-metal transport. Nickelates have become the newest frontier because they may bridge cuprate-like single-orbital physics and iron-based multiorbital physics, but sample quality, metastability, and definitive phase identification remain major bottlenecks.

**Key findings:**
- Cuprates remain the best-studied high-Tc platform, with strong evidence that charge order/stripes, pseudogap behavior, and superconductivity are deeply intertwined; the 2024 literature emphasizes charge correlations, collective charge excitations, and the role of disorder, magnetic fields, and lattice symmetry.
- The pseudogap is still the central unresolved cuprate problem, but recent experiments have sharpened the debate: shot-noise STM work argues that the pseudogap energy is tied to pair formation rather than purely to local charge order, while theory continues to offer competing pictures involving fluctuating PDW, antiferromagnetic domain walls, or fractionalized/pocket Fermi-surface states.
- PDW physics moved from speculative to increasingly concrete: multiple STM/x-ray/neutron studies support short-range or bulk PDW-related signatures in La-based cuprates, and theory now has microscopic models for fluctuating PDW states in strong-coupling regimes.
- Iron-based superconductors are comparatively more mature on the application side; the key frontier is high-field wire/tape engineering, where improved Ba-122 and related conductors are pushing engineering critical current density upward and making HTS deployment more realistic.
- Nickelates are the fastest-moving new unconventional family: infinite-layer nickelates now show superconductivity in thin films, while pressurized and, in some reports, ambient-pressure Ruddlesden–Popper nickelates show much higher Tc values. The field is still dominated by synthesis challenges, oxygen stoichiometry control, phase purity, and uncertainty about pairing symmetry and the true bulk nature of some superconducting signals.

**Open questions:**
- [0.75] What is the microscopic origin of the cuprate pseudogap: preformed pairs, fluctuating PDW, antiferromagnetic domain-wall physics, or some other fractionalized normal state?
- [0.70] Is PDW order a generic organizing principle of underdoped cuprates, or only a special feature of stripe-ordered compounds such as La-based systems?
- [0.85] What is the actual pairing glue in cuprates and nickelates: spin fluctuations, interlayer/orbital physics, phonons, or a mixed mechanism?
- [0.88] Can a fully convincing bulk, single-phase, ambient-pressure nickelate superconductor be stabilized, and if so, will its symmetry and normal state resemble cuprates or iron-based superconductors more closely?
- [0.80] What is the correct low-energy model for superconducting nickelates: effectively single-band, multiband, or a family-dependent hybrid description?
- [0.78] How universal are strange-metal transport and linear-in-temperature resistivity across high-Tc families, and are they causally linked to pairing or merely coexisting with it?

## The handoff: structured output as structured input

Here's the key idea: `survey` is a Python object. We can programmatically pick the most novel question, format it into a prompt, and hand it to the next agent. No string parsing, no regex, no hoping the LLM formatted its answer the way we expect.

In [6]:
best_question = max(survey.open_questions, key=lambda q: q.novelty)
print(f"Most novel (novelty={best_question.novelty:.2f}): {best_question.question}")

Most novel (novelty=0.88): Can a fully convincing bulk, single-phase, ambient-pressure nickelate superconductor be stabilized, and if so, will its symmetry and normal state resemble cuprates or iron-based superconductors more closely?


## Agent 2: the Theorist

The theorist takes the surveyor's landscape and the selected question, then produces a testable hypothesis.

In [7]:
theorist = Agent(
    name="theorist",
    prompt=(
        "You are a theoretical physicist. Given a survey and an open question, "
        "produce a precise, falsifiable hypothesis. Be concrete about the "
        "mechanism and what experiment would test it."
    ),
    config=config,
    response_schema=Hypothesis,
    tools=[exa_tool],
)

theorist_prompt = (
    f"Survey topic: {survey.topic}\n"
    f"Overview: {survey.overview}\n\n"
    f"Key findings:\n"
    + "\n".join(f"- {kf}" for kf in survey.key_findings)
    + f"\n\nFocus on this open question (novelty={best_question.novelty:.2f}):\n"
    f"{best_question.question}\n\n"
    f"Produce a falsifiable hypothesis with a concrete testable prediction."
)

hyp_reply = await theorist.ask(theorist_prompt)
hypothesis: Hypothesis = await hyp_reply.content()

In [8]:
lines = [
    "### Hypothesis",
    f"**Claim:** {hypothesis.claim}",
    f"**Mechanism:** {hypothesis.mechanism}",
    f"**Test:** {hypothesis.testable_prediction}",
    f"**Confidence:** {hypothesis.confidence:.2f}",
    f"**Addresses:** {hypothesis.source_question}",
]
display(Markdown("\n\n".join(lines)))

### Hypothesis

**Claim:** A bulk, single-phase, ambient-pressure nickelate superconductor can be stabilized only in a narrowly oxygen-ordered Ruddlesden–Popper nickelate with nominal d9-derived NiO2 planes, and its pairing will be predominantly sign-changing d-wave like the cuprates rather than s± like iron-based superconductors.

**Mechanism:** The mechanism is that long-range oxygen order removes the dominant pair-breaking disorder that currently masks intrinsic superconductivity in nickelates, while the resulting single-orbital Ni 3d_{x^2-y^2}-like low-energy manifold plus strong in-plane superexchange favors cuprate-like antinodal pairing. In this picture, the multiorbital/incipient-band physics that stabilizes s± in iron-based systems is subdominant, so once the phase is truly bulk and single-phase the leading instability should be a quasi-2D d-wave condensate tied to short-range antiferromagnetic correlations and a pseudogap-like normal state.

**Test:** A genuinely bulk, stoichiometric, single-crystal or epitaxial phase with no superconducting minority inclusions will show (i) a single thermodynamic superconducting anomaly in heat capacity and a full Meissner fraction at ambient pressure, (ii) a nodal gap structure with a sign change detected by phase-sensitive Josephson or quasiparticle-interference measurements, and (iii) a doping/temperature phase diagram with suppressed low-energy spectral weight and antiferromagnetic spin fluctuations resembling cuprates more than the multiband electron-hole nesting phenomenology of iron-based superconductors. The hypothesis is refuted if the only ambient-pressure superconducting nickelate states remain filamentary/inhomogeneous, or if a bulk single-phase sample instead shows a fully gapped s±-like response and multiband normal-state signatures characteristic of iron-based systems.

**Confidence:** 0.42

**Addresses:** Can a fully convincing bulk, single-phase, ambient-pressure nickelate superconductor be stabilized, and if so, will its symmetry and normal state resemble cuprates or iron-based superconductors more closely?

## The full pipeline in one view

Two agents, two typed schemas, one `max()` call to select the handoff, and standard string formatting for the prompt. No orchestration framework, no special message-passing protocol — the structured output *is* the protocol.

In [9]:
print(f"{survey.topic}")
print(f"  -> survey found {len(survey.open_questions)} open questions")
print(f"  -> selected: {best_question.question} (novelty={best_question.novelty:.2f})")
print(f"  -> hypothesis: {hypothesis.claim}")
print(f"  -> test: {hypothesis.testable_prediction}")
print(f"  -> confidence: {hypothesis.confidence:.2f}")

High-Tc superconductivity research
  -> survey found 6 open questions
  -> selected: Can a fully convincing bulk, single-phase, ambient-pressure nickelate superconductor be stabilized, and if so, will its symmetry and normal state resemble cuprates or iron-based superconductors more closely? (novelty=0.88)
  -> hypothesis: A bulk, single-phase, ambient-pressure nickelate superconductor can be stabilized only in a narrowly oxygen-ordered Ruddlesden–Popper nickelate with nominal d9-derived NiO2 planes, and its pairing will be predominantly sign-changing d-wave like the cuprates rather than s± like iron-based superconductors.
  -> test: A genuinely bulk, stoichiometric, single-crystal or epitaxial phase with no superconducting minority inclusions will show (i) a single thermodynamic superconducting anomaly in heat capacity and a full Meissner fraction at ambient pressure, (ii) a nodal gap structure with a sign change detected by phase-sensitive Josephson or quasiparticle-interference meas

## Notes

**Why typed handoffs matter.** Without `response_schema`, the surveyor returns a string. You'd have to parse it, hope the format is consistent, and pray the theorist interprets it correctly. With typed output, the surveyor *must* produce an `OpenQuestion` with a `novelty` float, or the schema validation rejects it. The handoff is just attribute access.

**The novelty score is the seed of recursion.** Right now we pick the single most novel question. In a later tutorial, we'll threshold on novelty and spawn a child agent for *every* question above 0.7 — that's how lionag2 does recursive depth. The schema makes it trivial: `[q for q in survey.open_questions if q.novelty >= 0.7]`.

**Agents are cheap.** An `Agent` in AG2 beta is a thin wrapper around a config + tools + system prompt. Creating a new one per pipeline stage costs nothing — no persistent process, no background thread. Think of them as typed function calls with LLM backends.

**Where's the stream?** We didn't pass an explicit `stream=` this time. Each `agent.ask()` creates a fresh `MemoryStream` by default. That's fine for independent pipeline stages. When we need agents to share history or react to each other's events, we'll wire them to the same stream.

## Up next

Day 4: **self-correction with schema retries**.

What happens when the theorist produces a hypothesis that fails validation — say, `confidence` is missing or the `claim` is too vague? AG2 beta's retry mechanism re-prompts the agent with the validation error, and the agent fixes its own output. No human in the loop, no second agent needed.

For day 1 of this series, see [01_get_started.ipynb](01_get_started.ipynb)  
For day 2, see [02_typed_findings.ipynb](02_typed_findings.ipynb)  
For more info on AG2 beta, read the [docs](https://docs.ag2.ai/)